# IKEAQueryGenerator 使用示例

本 Notebook 演示如何在不修改算法实现的前提下，直接使用 `src.skuas.IKEAQueryGenerator` 生成对抗式查询样例。内容包括环境检查、路径设置、最小输入构造、运行与保存结果。

In [1]:
# 1) 设置与环境检查
import sys, os, pathlib, platform, json, time, random
from datetime import datetime

In [2]:
# 2) 将工作区根目录加入 sys.path
import pathlib, sys
workspace = pathlib.Path('.').resolve()
if str(workspace) not in sys.path:
    sys.path.insert(0, str(workspace))
print("Workspace added to sys.path:", workspace)

Workspace added to sys.path: /WORK/PUBLIC/qiuhan4_work/zms/rag-llm


In [ ]:
# 3) 导入算法并打印版本信息
from src import IKEAQueryGenerator
from src import OpenAILLM

try:
    import torch
    print("torch:", torch.__version__)
except Exception as e:
    print("torch not available:", e)

print("IKEAQueryGenerator ready.")

In [ ]:
# 4) 构造最小化输入样例
import numpy as np

# 描述对象用于指导生成器（与 BBQ 保持风格一致）
description = {
    "intro": "This is a small demo corpus about general knowledge.",
    "type": "General Knowledge",
    "topic": "demo"
}

# 生成器参数
embed_model_name = "sentence-transformers/all-mpnet-base-v2"
adversarial_suffix = ""

print("demo description:", description)
print("embed model:", embed_model_name)

In [ ]:
# 5) 实现 Dummy LLM（离线可运行）并演示算法主入口
from src.interfaces import LLMManager
from configs import fiqa as cfg

llm_tool = OpenAILLM(model = cfg.tool_llm["model"], 
                    base_url = cfg.tool_llm["base_url"], 
                    api_key = cfg.tool_llm["api_key"], 
                    reasoning = cfg.tool_llm["reasoning"],
                    temperature = cfg.tool_llm["temperature"],
                    top_p = cfg.tool_llm["top_p"],
                    max_workers = 50)

ikea = IKEAQueryGenerator(
    description=description,
    llm=llm_tool,
    embed_mdl_name=embed_model_name,
    topic=description["topic"],
    adversarial_suffix=adversarial_suffix,
    device="cpu",
)

start = time.time()
queries = ikea.generate(attack_num=12)
elapsed = time.time() - start
print(f"Generated {len(queries)} queries in {elapsed:.3f}s")
print("Example queries (first 5):")
for i, q in enumerate(queries[:5], 1):
    print(f"{i:02d}. {q}")

In [ ]:
# 6) 结果校验与断言
assert queries is not None, "queries 应当非空"
assert isinstance(queries, list), "queries 应当是 list[str]"
assert len(queries) > 0, "至少应产生一条 query"
print("Assertions passed.")

In [ ]:
# 7) 打印关键结果摘要
from pprint import pprint

summary = {"num_queries": len(queries), "sample": queries[:3]}
pprint(summary)

In [ ]:
# 8) 保存结果到工作区根目录
import pathlib, json
outdir = pathlib.Path('outputs')
outdir.mkdir(exist_ok=True)
outfile = outdir / f"ikea_queries_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(outfile, 'w', encoding='utf-8') as f:
    json.dump({"queries": queries}, f, ensure_ascii=False, indent=2)
print("Saved to:", outfile)

In [ ]:
# 9) 复现性与性能计时
import numpy as np, random, time
random.seed(42)
np.random.seed(42)
try:
    import torch
    torch.manual_seed(42)
except Exception:
    pass

start = time.time()
_ = ikea.generate(attack_num=12)
print(f"Re-run generate() in {time.time() - start:.3f}s with fixed seeds.")

In [ ]:
# 10) 命令行等价用法（示例，不默认执行）
print("您可以在 VS Code 终端中运行如下命令（按需替换模型与 API 配置）：")
print("python -c \"from src.skuas import IKEAQueryGenerator; from src.llm import OpenAILLM; from src.utils import get_embed_model; print('CLI demo: import ok')\"")

In [ ]:
# 11) 清理临时文件（本示例无临时输入，示意保留最终 outputs）
from pathlib import Path
keep = list(Path('outputs').glob('ikea_queries_*.json'))
print("Keep outputs:")
for p in keep[:3]:
    print(" -", p)